# Data Preparation: Cleaning, Preprocessing and EDA (CSV + API)

This notebook demonstrates end-to-end data cleaning and exploratory analysis for both CSV datasets (movies, ratings, links, tags) and API product data. Outputs: cleaned/processed datasets, histograms, heatmaps, sparsity analysis, and saved reports/plots.

In [24]:
# Imports
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

# Ensure project root is on sys.path so `src` imports work when running the notebook
# Find project root by searching upward for a folder containing 'src'
project_root = Path.cwd()
found = False
for _ in range(10):
    if (project_root / 'src').exists():
        found = True
        break
    if project_root.parent == project_root:
        break
    project_root = project_root.parent

if not found:
    # Fallback: try common repository folder name
    candidate = Path.cwd() / 'RecoMart-Recommendation-Pipeline'
    if (candidate / 'src').exists():
        project_root = candidate
        found = True

if not found:
    raise RuntimeError(f'Could not locate project root with src/ from cwd={Path.cwd()}')

sys.path.insert(0, str(project_root.resolve()))

# Project modules
from src.config.config import CSV_FILES, API_RAW_DIR
from src.preparation.data_cleaner import DataCleaner
from src.preparation.data_preprocessor import DataPreprocessor
from src.preparation.exploratory_analysis import ExploratoryAnalyzer
from src.validation.api_validator import APIValidator, APIDataProfiler, get_default_product_schema, get_default_product_constraints
from src.utils.helper import save_json

# plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

In [25]:
# Output directories
from pathlib import Path

# Current directory
CURRENT_DIR = Path.cwd()

# Parent directory
ROOT = CURRENT_DIR.parent

OUT_PROCESSED = ROOT / 'data' / 'processed'
OUT_PLOTS = ROOT / 'data' / 'plots'
OUT_REPORTS = ROOT / 'data' / 'reports'
OUT_PROCESSED.mkdir(parents=True, exist_ok=True)
OUT_PLOTS.mkdir(parents=True, exist_ok=True)
OUT_REPORTS.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('Processed dir:', OUT_PROCESSED)
print('Plots dir:', OUT_PLOTS)
print('Reports dir:', OUT_REPORTS)

ROOT: f:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline
Processed dir: f:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\data\processed
Plots dir: f:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\data\plots
Reports dir: f:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\data\reports


## CSV Datasets: Cleaning & Preprocessing

In [ ]:
# Initialize helpers
cleaner = DataCleaner()
preprocessor = DataPreprocessor()
analyzer = ExploratoryAnalyzer(output_dir=OUT_PLOTS)

cleaning_reports = {}
preprocessing_reports = {}
eda_reports = {}

# 1) Clean CSVs using the DataCleaner
for name, path in CSV_FILES.items():
    print(f'Cleaning: {name} ->', path)
    df, report = cleaner.clean_csv(path, name)
    if df is None:
        print('  Cleaning failed for', name, report)
        continue
    cleaning_reports[name] = report

# 2) Preprocess cleaned datasets (encoding, normalization)
for name, df in cleaner.cleaned_datasets.items():
    print(f'Preprocessing: {name}')
    # Handle missing interactions for ratings as implicit zeros
    if name == 'ratings' and 'rating' in df.columns:
        df['rating'] = df['rating'].fillna(0)

    # For ratings, ensure rating is normalized later; encode text columns where appropriate
    df_processed, prep_report = preprocessor.preprocess_dataset(
        df, name, categorical_columns=None, numerical_columns=None, normalize_method='minmax'
    )
    preprocessing_reports[name] = prep_report

# Save processed CSVs
saved = preprocessor.save_processed_data(output_dir=OUT_PROCESSED)
print('Saved processed files:', saved)

2026-07-19 03:19:01,893 | INFO | Cleaning dataset: movies


Cleaning: movies -> F:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\dataset\movies.csv


2026-07-19 03:19:02,177 | INFO | Cleaning completed for movies: 0 rows dropped
2026-07-19 03:19:02,177 | INFO | Cleaning dataset: ratings
2026-07-19 03:19:02,258 | INFO | Cleaning completed for ratings: 0 rows dropped
2026-07-19 03:19:02,258 | INFO | Cleaning dataset: links
2026-07-19 03:19:02,313 | INFO | Cleaning completed for links: 8 rows dropped
2026-07-19 03:19:02,313 | INFO | Cleaning dataset: tags
2026-07-19 03:19:02,324 | INFO | Cleaning completed for tags: 0 rows dropped
2026-07-19 03:19:02,324 | INFO | Preprocessing dataset: movies


Cleaning: ratings -> F:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\dataset\ratings.csv
Cleaning: links -> F:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\dataset\links.csv
Cleaning: tags -> F:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\dataset\tags.csv
Preprocessing: movies


2026-07-19 03:19:02,389 | INFO | Preprocessing completed for movies
2026-07-19 03:19:02,403 | INFO | Preprocessing dataset: ratings
2026-07-19 03:19:02,426 | INFO | Preprocessing completed for ratings
2026-07-19 03:19:02,426 | INFO | Preprocessing dataset: links
2026-07-19 03:19:02,440 | INFO | Preprocessing completed for links
2026-07-19 03:19:02,440 | INFO | Preprocessing dataset: tags
2026-07-19 03:19:02,459 | INFO | Preprocessing completed for tags
2026-07-19 03:19:02,511 | INFO | Saved processed data: f:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\data\processed\movies_processed.csv


Preprocessing: ratings
Preprocessing: links
Preprocessing: tags


2026-07-19 03:19:03,142 | INFO | Saved processed data: f:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\data\processed\ratings_processed.csv
2026-07-19 03:19:03,192 | INFO | Saved processed data: f:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\data\processed\links_processed.csv
2026-07-19 03:19:03,214 | INFO | Saved processed data: f:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\data\processed\tags_processed.csv


Saved processed files: {'movies': WindowsPath('f:/DM4MLPROJECT/DM4ML-Group12/RecoMart-Recommendation-Pipeline/data/processed/movies_processed.csv'), 'ratings': WindowsPath('f:/DM4MLPROJECT/DM4ML-Group12/RecoMart-Recommendation-Pipeline/data/processed/ratings_processed.csv'), 'links': WindowsPath('f:/DM4MLPROJECT/DM4ML-Group12/RecoMart-Recommendation-Pipeline/data/processed/links_processed.csv'), 'tags': WindowsPath('f:/DM4MLPROJECT/DM4ML-Group12/RecoMart-Recommendation-Pipeline/data/processed/tags_processed.csv')}


In [27]:
# 3) EDA for CSV datasets: distributions, heatmaps, sparsity (for ratings)
for name, df in preprocessor.processed_datasets.items():
    print('EDA for:', name)
    analysis = analyzer.analyze_dataset(df, name)
    eda_reports[name] = analysis

    # Distribution plot (numerical)
    dist = analyzer.plot_distributions(df, name, save=True)
    print('  distribution plot ->', dist)

    # Heatmap of correlations when applicable
    heat = analyzer.plot_heatmap(df, name, save=True)
    print('  heatmap ->', heat)

    # Categorical distributions
    catp = analyzer.plot_categorical_distributions(df, name, save=True)
    print('  categorical plot ->', catp)

    # Ratings: create interaction matrix and sparsity plot
    if name == 'ratings':
        # interaction matrix created from processed ratings (userId, movieId, rating)
        interaction_matrix = preprocessor.create_interaction_matrix(df, user_col='userId', item_col='movieId', rating_col='rating')
        sparsity_plot = analyzer.plot_sparsity(interaction_matrix, dataset_name='interactions', save=True)
        print('  sparsity plot ->', sparsity_plot)

# Save summary JSON reports
save_json(cleaning_reports, OUT_REPORTS / 'cleaning_reports.json')
save_json(preprocessing_reports, OUT_REPORTS / 'preprocessing_reports.json')
save_json(eda_reports, OUT_REPORTS / 'eda_reports.json')
print('CSV EDA complete. Reports saved to', OUT_REPORTS)

2026-07-19 03:19:03,242 | INFO | Analyzing dataset: movies


EDA for: movies


2026-07-19 03:19:04,395 | INFO | Saved distribution plot: f:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\data\plots\movies_distributions.png


  distribution plot -> f:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\data\plots\movies_distributions.png


2026-07-19 03:19:04,868 | INFO | Saved heatmap: f:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\data\plots\movies_heatmap.png
2026-07-19 03:19:04,869 | WARNING | No categorical columns found in movies
2026-07-19 03:19:04,870 | INFO | Analyzing dataset: ratings


...

  heatmap -> f:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\data\plots\links_heatmap.png
  categorical plot -> None
EDA for: tags


2026-07-19 03:19:11,085 | INFO | Saved distribution plot: f:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\data\plots\tags_distributions.png


  distribution plot -> f:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\data\plots\tags_distributions.png


2026-07-19 03:19:11,473 | INFO | Saved heatmap: f:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\data\plots\tags_heatmap.png
2026-07-19 03:19:11,473 | WARNING | No categorical columns found in tags


  heatmap -> f:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\data\plots\tags_heatmap.png
  categorical plot -> None
CSV EDA complete. Reports saved to f:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\data\reports


## API Data: Ingest, Validate, Clean, Preprocess, and EDA

In [28]:
# Find latest API products.json
api_files = list(Path(API_RAW_DIR).rglob('products.json'))
if not api_files:
    raise FileNotFoundError('No products.json found under API raw dir')
latest_api = max(api_files, key=lambda p: p.stat().st_mtime)
print('Using API file:', latest_api)

# Validate API data and profile
validator = APIValidator()
schema = get_default_product_schema()
constraints = get_default_product_constraints()
valid_records, api_report = validator.validate_json_file(latest_api, schema, constraints)
print('API validation: valid records ->', len(valid_records) if valid_records else 0)

profiler = APIDataProfiler()
api_profile = profiler.profile_api_data(valid_records, 'products')
save_json(api_report, OUT_REPORTS / 'api_validation_report.json')
save_json(api_profile, OUT_REPORTS / 'api_profile.json')
print('API reports saved to', OUT_REPORTS)

2026-07-19 03:19:11,562 | INFO | Validating API data: products.json
2026-07-19 03:19:11,626 | INFO | Validation completed: 30 valid, 0 invalid out of 30
2026-07-19 03:19:11,626 | INFO | Profiling API dataset: products


Using API file: F:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\data\raw\api\products\2026\07\19\030815\products.json
API validation: valid records -> 30
API reports saved to f:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\data\reports


In [32]:
# Convert valid API records to DataFrame and apply cleaning/preprocessing similar to CSVs
if valid_records:
    df_api = pd.DataFrame(valid_records)

    # Basic cleaning: fill numeric missing values (mean) and categorical missing values ('Unknown')
    numeric_cols = ['price', 'rating', 'discountPercentage', 'stock']
    for col in numeric_cols:
        if col in df_api.columns:
            if pd.api.types.is_numeric_dtype(df_api[col]):
                df_api[col].fillna(df_api[col].mean(), inplace=True)
            else:
                df_api[col].fillna(0, inplace=True)

    for col in ['title', 'brand', 'category', 'description']:
        if col in df_api.columns:
            df_api[col].fillna('Unknown', inplace=True)

    # Preprocess API dataframe: encode categorical and normalize numeric columns
    api_preprocessor = DataPreprocessor()
    categorical_cols = [c for c in ['brand', 'category'] if c in df_api.columns]
    numerical_cols = [c for c in ['price', 'rating', 'discountPercentage', 'stock'] if c in df_api.columns]

    df_api_processed, api_prep_report = api_preprocessor.preprocess_dataset(
        df_api, 'products', categorical_columns=categorical_cols, numerical_columns=numerical_cols, normalize_method='minmax'
    )

    # Save processed API dataset and EDA
    api_out_file = OUT_PROCESSED / 'products_processed.csv'
    df_api_processed.to_csv(api_out_file, index=False)
    print('Saved API processed dataset ->', api_out_file)

    # EDA: distributions and heatmap
    analyzer.plot_distributions(df_api_processed, 'products', save=True)
    analyzer.plot_heatmap(df_api_processed, 'products', save=True)
    save_json(api_prep_report, OUT_REPORTS / 'api_preprocessing_report.json')
    print('API preprocessing & EDA complete. Reports saved to', OUT_REPORTS)
else:
    print('No valid API records to process')

C:\Users\ANJALI\AppData\Local\Temp\ipykernel_19308\2428916959.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_api[col].fillna(df_api[col].mean(), inplace=True)
C:\Users\ANJALI\AppData\Local\Temp\ipykernel_19308\2428916959.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.


Saved API processed dataset -> f:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\data\processed\products_processed.csv


2026-07-19 03:30:45,347 | INFO | Saved distribution plot: f:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\data\plots\products_distributions.png
2026-07-19 03:30:46,118 | INFO | Saved heatmap: f:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\data\plots\products_heatmap.png


API preprocessing & EDA complete. Reports saved to f:\DM4MLPROJECT\DM4ML-Group12\RecoMart-Recommendation-Pipeline\data\reports


## Summary
- Processed CSV datasets saved to `data/processed`
- API processed dataset saved to `data/processed/products_processed.csv` (if API records were valid)
- Plots (histograms, heatmaps, sparsity) saved to `data/plots`
- JSON reports saved to `data/reports`

Next steps: use the processed datasets in feature engineering and model training pipelines; optionally tune missing-value strategies or categorical encoding methods (one-hot vs label encoding) depending on downstream model requirements.